In [38]:
import argparse
import os

import pandas as pd
import pickle
from tqdm import tqdm
from copy import deepcopy

import numpy as np
import torch
from torch import optim
from torch.utils.tensorboard import SummaryWriter

from dp_fw import dp_fw

import sys

In [39]:
def load_data_all(dataset, s):
    print(dataset)
    preprocess_file = './data/'+dataset+'.pkl'
    full_M_file = './data/'+dataset+'/matrix.pt'
    if dataset == 'random':
        mat1 = torch.randn(100,1)
        mat2 = torch.randn(1, 5)
        matrix = mat1 @ mat2
        return mat1, mat2, matrix

    elif dataset == 'ml-1m':
        # read table
        ratings_title = ['UserID','MovieID', 'ratings', 'timestamps']
        ratings = pd.read_table(data_path+'ml-1m/ratings.dat', sep='::', header=None, names=ratings_title, engine = 'python')
        ratings = ratings.filter(regex='UserID|MovieID|ratings')

        # sample
        unique_ids = ratings['MovieID'].unique()
        selected_ids = np.random.choice(unique_ids, size=s, replace=False)
        selected_data = ratings[ratings['MovieID'].isin(selected_ids)]

        #selected_data = torch.tensor(selected_data.values)
        matrix_df = selected_data.pivot_table(index='UserID', columns='MovieID', values='ratings', aggfunc='first')
        matrix_df = matrix_df.fillna(0)
        matrix = torch.tensor(matrix_df.values)

        return matrix
    
    elif dataset == 'netflix':
        df = pd.read_csv(
            "./data/netflix/data.csv",
            names=["movie_id", "user_id", "rating", "date"],
            parse_dates=["date"],
            encoding="ISO-8859-1",
            engine="python",
        )
        data = df[["movie_id", "user_id", "rating"]]
        # sample
        unique_ids = data['movie_id'].unique()
        selected_ids = np.random.choice(unique_ids, size=s, replace=False)
        selected_data = data[data['movie_id'].isin(selected_ids)]

        #selected_data = torch.tensor(selected_data.values)
        matrix_df = selected_data.pivot_table(index='user_id', columns='movie_id', values='rating', aggfunc='first')
        matrix_df = matrix_df.fillna(0)
        matrix = torch.tensor(matrix_df.values)

        return matrix
    
    elif dataset == 'gene':
        M = load_data_gene()

mat1, mat2, M = load_data_all('random', s=150)

print(f"original user: {M.shape[0]} item: {M.shape[1]}")
M = M.float()

random
original user: 100 item: 5


In [40]:
def top_r_svd(A, r):
    V, S, Vt = torch.linalg.svd(A, full_matrices=False)

    Vr = V[:, :r]  
    Sr = S[:r]   
    Vtr = Vt[:r, :]
    return Vr, Sr, Vtr

In [41]:
r = mat2.shape[0]
epochs = 100
alpha = 0.1
p = 0.05
noise = 0.01
dataset = 'random'


In [42]:
M = M[torch.count_nonzero(M, dim=1)>r]
print(f"used user: {M.shape[0]} item: {M.shape[1]}")
# mask

mask = torch.bernoulli(torch.full((M.shape[0],), p)).bool()
observed_M = deepcopy(M)
observed_M[mask] = 0
u, d, v = top_r_svd(observed_M, r)
#observed_M = observed_M[~mask]

miss_num = mask.sum()
print(f"masked user: {miss_num}, ratio: {miss_num/M.shape[0]:.2f}")

# find entries
non_zero_indices = M[mask].nonzero(as_tuple=False)
miss_entries = [tuple(idx.tolist()) for idx in non_zero_indices]
miss_users = torch.tensor(range(M.shape[0]))[mask]
count_user = torch.count_nonzero(M[mask], dim=1).float()
print(count_user.median())

# opt
#X = dp_sgd_mannual(observed_M, args).detach()
writer = SummaryWriter('runs/'+dataset)
# parameters
d1, d2 = observed_M.shape
#eta = args.eta
eta = 1 / np.sqrt(epochs)
alpha = alpha
# optimize


norms = torch.norm(M, dim=0)
maxl2norm = norms.max()
observed_M_norm = M / (norms + 1e-8)
#observed_M_norm = M 
MTM = torch.mm(observed_M_norm.t(), observed_M_norm)
scale_diag = (p-1) / (p**2) * torch.diag(MTM).diag()
A = scale_diag + MTM

X = torch.randn(d2, d2, requires_grad=True)
#with torch.no_grad():
#    X = torch.tensor(A, requires_grad=True)
optimizer = optim.SGD([X], lr=eta)
#init_error = recover_M(M, X, miss_users, r)

loop = tqdm(range(epochs))
for i in loop:
    #XTX = torch.mm(X.t(), X)
    #loss = ((XTX - A)**2).mean()
    loss = ((X - A)**2).mean()
    U, D, Vt = torch.linalg.svd(X, full_matrices=False)
    #loss = D.sum()
    loss.backward()
    
    #U, D, Vt = top_r_svd(X, r)
    #U, _ = top_r_eigen(X, r)
    #print(X.grad)

    noise_matrix = torch.normal(mean=0, std=noise, size=(d2, d2))

    X.grad = X.grad + alpha * U @ Vt + noise_matrix
    """
    with torch.no_grad():
        X.data += -eta*X.grad
    """
    optimizer.step()
    X.grad.zero_()
    #print(loss.item())
    writer.add_scalar('Loss', loss.item(), i)
    loop.set_postfix_str('In epoch {}, loss: {:.3f})'.format(i, loss.item()))
X = X * ((norms + 1e-8)**2)

#print(torch.norm(X))
#print(torch.norm(mat2.t() @ mat2))
#print(X - mat2.t() @ mat2)

u, d, v = top_r_svd(mat2, r)
print(d)
#X = M.t() @ M
#u, d, v = torch.linalg.svd(mat2)
m2 = v.t() @ v
ux, dx, vx = top_r_svd(X, r)
#ux, dx, vx = torch.linalg.svd(X)
xx = vx.t() @ vx
xxx = ux @ dx.diag() @ vx
print(vx)
print(v)
print(torch.norm(xx))
print(torch.norm(m2))
print((xx-m2))
print(xx.mean())
print(torch.allclose(xx, m2, atol=1e-03, rtol=1e-03))


used user: 100 item: 5
masked user: 3, ratio: 0.03
tensor(5.)


100%|██████████| 100/100 [00:00<00:00, 645.61it/s, In epoch 99, loss: 5899.313)]

tensor([1.3222])
tensor([[ 9.9996e-01, -8.8795e-03,  6.2169e-04,  4.2406e-04,  7.0831e-04]],
       grad_fn=<SliceBackward0>)
tensor([[-0.6788, -0.5871,  0.1949,  0.3234,  0.2278]])
tensor(1., grad_fn=<LinalgVectorNormBackward0>)
tensor(1.0000)
tensor([[ 0.5391, -0.4074,  0.1329,  0.2200,  0.1553],
        [-0.4074, -0.3446,  0.1144,  0.1899,  0.1337],
        [ 0.1329,  0.1144, -0.0380, -0.0630, -0.0444],
        [ 0.2200,  0.1899, -0.0630, -0.1046, -0.0737],
        [ 0.1553,  0.1337, -0.0444, -0.0737, -0.0519]], grad_fn=<SubBackward0>)
tensor(0.0394, grad_fn=<MeanBackward0>)
False
